In [16]:
import warnings
warnings.filterwarnings("ignore")

In [17]:
TICKERS = [
    "AAPL","AXP","KO","BAC","CVX",
    "MCO","OXY","CB","KHC","GOOGL"
]

START_DATE = "2021-01-01"
END_DATE = "2026-06-01"

FORECAST_HORIZON = 120  # ~6 months

TOP_K = 5

In [18]:
import yfinance as yf
print("Downloading data...")

raw = yf.download(
    TICKERS,
    start=START_DATE,
    end=END_DATE,
    auto_adjust=True
)

[**************        30%                       ]  3 of 10 completed

[*********************100%***********************]  10 of 10 completed


In [19]:
import pandas as pd
from ta.trend import SMAIndicator, EMAIndicator, MACD
from ta.momentum import RSIIndicator
from ta.volatility import BollingerBands

def create_features(df):
    df = df.copy()

    # SMA
    df["sma20"] = SMAIndicator(df["Close"], window=20).sma_indicator()
    df["sma50"] = SMAIndicator(df["Close"], window=50).sma_indicator()

    # EMA
    df["ema20"] = EMAIndicator(df["Close"], window=20).ema_indicator()

    # RSI
    df["rsi"] = RSIIndicator(df["Close"], window=14).rsi()

    # MACD
    macd = MACD(df["Close"])
    df["macd"] = macd.macd()
    df["macd_signal"] = macd.macd_signal()

    # Bollinger Bands
    bb = BollingerBands(df["Close"])
    df["bb_high"] = bb.bollinger_hband()
    df["bb_low"] = bb.bollinger_lband()

    # Returns
    df["returns"] = df["Close"].pct_change()

    # Volatility
    df["volatility"] = df["returns"].rolling(20).std()

    df = df.dropna()

    return df

In [20]:
def directional_accuracy(y_true, y_pred):
    actual_direction = np.sign(np.diff(y_true))
    pred_direction = np.sign(np.diff(y_pred))

    correct = (actual_direction == pred_direction).sum()
    total = len(actual_direction)

    return correct / total

In [21]:
from neuralforecast import NeuralForecast
from neuralforecast.models import PatchTST
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.preprocessing import StandardScaler
from matplotlib.backends.backend_pdf import PdfPages
from tqdm import tqdm

all_forecasts = []
plot_pdf = PdfPages("forecast_report.pdf")
for ticker in tqdm(TICKERS, total=len(TICKERS)):
    df = pd.DataFrame({
        "Date": raw.index,
        "Close": raw["Close"][ticker].values,
        "Volume": raw["Volume"][ticker].values
    })

    df = create_features(df)
    nf_df = pd.DataFrame({
        "unique_id": ticker,
        "ds": pd.to_datetime(df["Date"]),
        "y": df["Close"]
    })

    train_size = len(nf_df) - FORECAST_HORIZON

    train_df = nf_df.iloc[:train_size]
    test_df = nf_df.iloc[train_size:]

    model = PatchTST(
        h=FORECAST_HORIZON,
        input_size=252,
        patch_len=16,
        stride=8,
        hidden_size=64,
        n_heads=4,
        learning_rate=1e-3,
        max_steps=300,
        batch_size=32
    )

    nf = NeuralForecast(
        models=[model],
        freq='D'
    )

    # Train
    nf.fit(df=train_df)

    forecast = nf.predict()
    preds = forecast["PatchTST"].values

    # evaluation
    y_true = test_df["y"].values[:len(preds)]
    mae = mean_absolute_error(y_true, preds)
    rmse = np.sqrt(mean_squared_error(y_true, preds))
    dstat = directional_accuracy(y_true, preds)

    # calculation
    current_price = df["Close"].iloc[-1]
    forecast_price = preds[-1]
    volatility = df["volatility"].iloc[-1]
    expected_return = ((forecast_price - current_price) / current_price) * 100
    all_forecasts.append({
        "Ticker": ticker,
        "CurrentPrice": current_price,
        "ForecastPrice": forecast_price,
        "ExpectedReturn": expected_return,
        "MAE": mae,
        "RMSE": rmse,
        "Dstat": dstat,
        "Volatility": volatility
    })


    plt.figure(figsize=(12,6))
    plt.plot(test_df["ds"][:len(preds)], y_true, label="Actual")
    plt.plot(test_df["ds"][:len(preds)], preds, label="Forecast")

    plt.title(f"{ticker} Forecast")
    plt.xlabel("Date")
    plt.ylabel("Price")
    plt.legend()
    plt.grid(True)

    plot_pdf.savefig()
    plt.close()

  0%|                                                                                                                                                           | 0/10 [00:00<?, ?it/s]Seed set to 1
GPU available: False, used: False
TPU available: False, using: 0 TPU cores

  | Name         | Type              | Params | Mode 
-----------------------------------------------------------
0 | loss         | MAE               | 0      | train
1 | padder_train | ConstantPad1d     | 0      | train
2 | scaler       | TemporalNorm      | 0      | train
3 | model        | PatchTST_backbone | 391 K  | train
-----------------------------------------------------------
391 K     Trainable params
3         Non-trainable params
391 K     Total params
1.565     Total estimated model params size (MB)
90        Modules in train mode
0         Modules in eval mode


Sanity Checking: |                                                                                            …

Training: |                                                                                                   …

Validation: |                                                                                                 …

Validation: |                                                                                                 …

...

Validation: |                                                                                                 …

Validation: |                                                                                                 …

`Trainer.fit` stopped: `max_steps=300` reached.
GPU available: False, used: False
TPU available: False, using: 0 TPU cores


Predicting: |                                                                                                 …

100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 10/10 [56:16<00:00, 337.69s/it]


In [22]:
result_df = pd.DataFrame(all_forecasts)
result_df = result_df.sort_values(
    by="ExpectedReturn",
    ascending=False
)

print("\n==============================")
print("ALL STOCK RANKING")
print("==============================")

print(result_df)

top5 = result_df.head(TOP_K).copy()
total_return = top5["ExpectedReturn"].sum()
top5["Score"] = (top5["ExpectedReturn"] * top5["Dstat"]) / top5["Volatility"]
top5["Weight"] = (top5["Score"]/ top5["Score"].sum())


TOTAL_FUNDS = 1_000_000_000_000
top5["Allocation"] = (top5["Weight"] * TOTAL_FUNDS)
top5["ProjectedProfit"] = (top5["Allocation"] * top5["ExpectedReturn"] / 100)

print("\n==============================")
print("TOP 5 PORTFOLIO")
print("==============================")

print(top5)

result_df.to_csv("all_stock_forecast.csv", index=False)
top5.to_csv("top5_portfolio.csv", index=False)

print("\nSaved:")
print("- all_stock_forecast.csv")
print("- top5_portfolio.csv")


ALL STOCK RANKING
  Ticker  CurrentPrice  ForecastPrice  ExpectedReturn        MAE       RMSE  \
1    AXP    316.470001     438.999969       38.717720  55.610298  69.312603   
3    BAC     51.333282      64.608002       25.859868   7.077185   8.544455   
5    MCO    453.250000     515.220642       13.672508  54.864798  64.626348   
8    KHC     23.582586      22.642996       -3.984255   0.839812   0.984219   
7     CB    311.730011     296.849365       -4.773569  27.245141  31.063475   
0   AAPL    312.059998     293.591766       -5.918167  18.704860  21.440115   
2     KO     79.010002      69.922264      -11.502010   7.616015   8.673265   
6    OXY     56.630001      47.578201      -15.984107   7.507195   9.488707   
9  GOOGL    380.112946     310.772430      -18.242082  50.935228  60.028278   
4    CVX    182.460007     136.912552      -24.962980  33.427054  37.590570   

      Dstat  Volatility  
1  0.436975    0.008929  
3  0.453782    0.013873  
5  0.487395    0.015463  
8  0.50

In [23]:
total_profit = top5["ProjectedProfit"].sum()

In [24]:
print("\n==============================")
print("TOTAL PROJECTED PROFIT")
print("==============================")
print(f"Total Profit: Rp {total_profit:,.0f}")


TOTAL PROJECTED PROFIT
Total Profit: Rp 363,333,798,049


In [25]:
portfolio_return = top5["ExpectedReturn"].dot(top5["Weight"])
print(f"Portfolio Return: {portfolio_return:.2f}%")

Portfolio Return: 36.33%


In [26]:
import pandas as pd

df_stock = pd.read_csv("top5_portofolio.csv")
tickers = df["Ticker"].tolist()

FileNotFoundError: [Errno 2] No such file or directory: 'top5_portofolio'

In [27]:
results = []
for ticker in tickers:
    model = PatchTST(
        h=FORECAST_HORIZON,
        input_size=252,
        patch_len=16,
        stride=8,
        hidden_size=64,
        n_heads=4,
        learning_rate=1e-3,
        max_steps=300,
        batch_size=32
    )
    
    nf = NeuralForecast(
        models=[model],
        freq='D'
    )
    
    # TRAIN ALL HISTORICAL DATA
    nf.fit(df=nf_df)
    
    # PREDICT FUTURE
    future_forecast = nf.predict()
    
    # =====================================
    # CURRENT PRICE
    # =====================================
    
    current_price = nf_df["y"].iloc[-1]
    
    # =====================================
    # FUTURE FORECAST PRICE
    # =====================================
    
    forecast_price = future_forecast["PatchTST"].iloc[-1]
    
    # =====================================
    # EXPECTED RETURN
    # =====================================
    
    expected_return = (
        (forecast_price - current_price)
        / current_price
    ) * 100
    
    # =====================================
    # SAVE RESULT
    # =====================================
    
    results.append({
        "Ticker": ticker,
        "CurrentPrice": current_price,
        "ForecastPrice": forecast_price,
        "ExpectedReturn": expected_return
    })

NameError: name 'tickers' is not defined

In [ ]:
EXPORT_TOP5_FORECAST = "./top5_forecast.csv"

In [ ]:
import pandas as pd

result_df = pd.DataFrame(results)
result_df = result_df.sort_values(
    by="ExpectedReturn",
    ascending=False
)
top5 = result_df.head(5).copy()
top5["Weight"] = (top5["ExpectedReturn"] / top5["ExpectedReturn"].sum())

TOTAL_FUNDS = 1_000_000_000_000
top5["Allocation"] = (top5["Weight"] * TOTAL_FUNDS)
top5["ProjectedProfit"] = (top5["Allocation"] * top5["ExpectedReturn"]/ 100)

top5.to_csv(EXPORT_TOP5_FORECAST, index=False)

total_profit = top5["ProjectedProfit"].sum()
print(top5)

print("\n====================")
print("TOTAL PROFIT")
print("====================")

print(f"Rp {total_profit:,.0f}")